In [ ]:
# lesson 2 : Pydantic basics

In [1]:
# imports 
from pydantic import BaseModel, ValidationError, EmailStr
import json

In [2]:
# Define UserInput Pydantic model and populate with data 
class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str

In [3]:
# create a model instance 
user_input = UserInput(
    name="Joe User",
    email="joe.user@example.com",
    query="I forgot my password"
)
print(user_input)

name='Joe User' email='joe.user@example.com' query='I forgot my password'


In [4]:
# attempt to create another model instance with invalid email - validation error
user_input_2 = UserInput(
    name="Joe User",
    email="not-an-email",
    query="I forgot my password"
)
print(user_input_2) 

ValidationError: 1 validation error for UserInput
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]

In [5]:
# define a function to handle user input validation safely
def validate_user_input(input_data):
    try:
        # attempt to create a UserInstance model instance
        user_input = UserInput(**input_data)
        print("Valid user input created:")
        print(user_input.model_dump_json(indent=2)) # method converts a Pydantic model instance directly into a JSON-encoded string
        return user_input
    except ValidationError as e:
        print("Validation error occurred:")
        for err in e.errors():
            print(f" - {err['loc'][0]}: {err['msg']}")
        return None

In [6]:
# create model instance using validate_user_input()
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com",
    "query" : "I forgot my password"
}
user_input = validate_user_input(input_data)
print(type(user_input))

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password"
}
<class '__main__.UserInput'>


In [7]:
# attempt to create model instance with missing input data
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com"
}
user_input = validate_user_input(input_data)

Validation error occurred:
 - query: Field required


In [8]:
# update model with additional fields : 
from pydantic import Field
from typing import Optional
from datetime import date

class UserInput(BaseModel):
    name: str
    email: EmailStr
    query: str
    order_id: Optional[int] = Field(
        None,
        description="5 digit number (cannot start with 0)",
        ge=10000,
        le=99999
    )
    purchase_date: Optional[date] = None

In [9]:
# define a dict with required fields only : 
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com",
    "query" : "I forgot my password"
}

# validate inp 
user_input = validate_user_input(input_data)
# null is json format (model_dump_json)
# json representation

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I forgot my password",
  "order_id": null,
  "purchase_date": null
}


In [22]:
user_input
# python representation

UserInput(name='Joe User', email='joe.user@example.com', query='I forgot my password', order_id=None, purchase_date=None)

In [10]:
# define a payload with all fields, including additional
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com",
    "query" : "I forgot my password",
    "query" : f"""I bought a laptop carrying case and it turned out to be
               the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date": date(2026,3,30),
    "system_message" : "logging status regarding order processing...",
    "iteartion": 1
}

# validate user inp
user_input = validate_user_input(input_data)

# Pydantic will ignore extra fields that do not exist in the data model
# data is printed as a str because of json format

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be\n               the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2026-03-30"
}


In [11]:
user_input
# python prints datetime obj

UserInput(name='Joe User', email='joe.user@example.com', query='I bought a laptop carrying case and it turned out to be\n               the wrong size. I need to return it.', order_id=12345, purchase_date=datetime.date(2026, 3, 30))

In [12]:
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com",
    "query" : "I forgot my password",
    "query" : f"""I bought a laptop carrying case and it turned out to be
               the wrong size. I need to return it.""",
    "order_id": 12345,
    "purchase_date":"2026-03-30",
}
user_input = validate_user_input(input_data)
# str is valid data type for purchase date

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be\n               the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2026-03-30"
}


In [13]:
user_input
# python converted str to datetime

UserInput(name='Joe User', email='joe.user@example.com', query='I bought a laptop carrying case and it turned out to be\n               the wrong size. I need to return it.', order_id=12345, purchase_date=datetime.date(2026, 3, 30))

In [14]:
# Data type coercion happens automatically for certain data types by Pydantic. Another example, int - str
# you can turn off this feature.
# order_id is defined as int but you can have string in its place.
input_data = {
    "name" : "Joe User",
    "email" : "joe.user@example.com",
    "query" : "I forgot my password",
    "query" : f"""I bought a laptop carrying case and it turned out to be
               the wrong size. I need to return it.""",
    "order_id": "12345",
    "purchase_date":"2026-03-30",
}
user_input = validate_user_input(input_data)

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I bought a laptop carrying case and it turned out to be\n               the wrong size. I need to return it.",
  "order_id": 12345,
  "purchase_date": "2026-03-30"
}


In [15]:
# str cannot be coerced from int
# name is defined as an str
input_data = {
    "name" : 999,
    "email" : "joe.user@example.com",
    "query" : "I forgot my password",
    "query" : f"""I bought a laptop carrying case and it turned out to be
               the wrong size. I need to return it.""",
    "order_id": "12345",
    "purchase_date":"2026-03-30",
}
user_input = validate_user_input(input_data)

Validation error occurred:
 - name: Input should be a valid string


In [32]:
# Json data coming from the front end application : 
json_data = '''
{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "I bought a keyboard and mouse and was overcharged.",
    "order_id": 12345,
    "purchase_date": "2025-12-31"
}
'''

# parse json str into python dict
inp_data = json.loads(json_data)
print("Parsed JSON:", inp_data)

Parsed JSON: {'name': 'Joe User', 'email': 'joe.user@example.com', 'query': 'I bought a keyboard and mouse and was overcharged.', 'order_id': 12345, 'purchase_date': '2025-12-31'}


In [33]:
# validate the customer support data from JSON 
user_input = validate_user_input(inp_data)

Valid user input created:
{
  "name": "Joe User",
  "email": "joe.user@example.com",
  "query": "I bought a keyboard and mouse and was overcharged.",
  "order_id": 12345,
  "purchase_date": "2025-12-31"
}


In [16]:
# processing json with alternative data formats : 
# order id cannot start with 0 
json_data = '''
{
    "name": "Joe User",
    "email": "joe.user@example.com",
    "query": "My account has been locked for some reason.",
    "order_id": "01234",
    "purchase_date": "2025-12-31"
}
'''

# parse json str into python dict
inp_data = json.loads(json_data)
print("Parsed JSON:", inp_data)


Parsed JSON: {'name': 'Joe User', 'email': 'joe.user@example.com', 'query': 'My account has been locked for some reason.', 'order_id': '01234', 'purchase_date': '2025-12-31'}


In [17]:
# validate the customer support data from JSON 
user_input = validate_user_input(inp_data)

Validation error occurred:
 - order_id: Input should be greater than or equal to 10000


In [38]:
# Rather than doing a two step process :
# 1) create json obj from str
# 2) validate json

# use built in model_validate_json

user_input = UserInput.model_validate_json(json_data)
print(user_input.model_dump_json(indent=2))

ValidationError: 1 validation error for UserInput
order_id
  Input should be greater than or equal to 10000 [type=greater_than_equal, input_value='01234', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/greater_than_equal